# Training ConvNeXt-Tiny for Bee Orientation Regression


### 1. Imports


In [ ]:
import os
import sys
import warnings
import logging
from pathlib import Path
from common.bee_dataset import BeeOrientationDataset, index_crops, split_by_trajectory

if "OMP_NUM_THREADS" not in os.environ:
    _cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
    os.environ["OMP_NUM_THREADS"] = str(_cpus)

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from tqdm import tqdm
import matplotlib.pyplot as plt

for _cand in (Path.cwd().parent, Path.cwd() / "src", Path.cwd()):
    if (_cand / "common" / "bee_dataset.py").exists():
        sys.path.insert(0, str(_cand.resolve()))
        break

warnings.filterwarnings("ignore", message=".*record_function.*")
logging.getLogger("torch._inductor").setLevel(logging.ERROR)
logging.getLogger("torch._logging").setLevel(logging.ERROR)

print("Imports OK")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU count       : {torch.cuda.device_count()}")
print(f"OMP_NUM_THREADS : {os.environ['OMP_NUM_THREADS']}")

### 2. Distributed / Device Setup

To launch DDP on SLURM:

```bash
torchrun --standalone --nproc_per_node=$SLURM_GPUS_ON_NODE train_ddp.py
```


In [ ]:
# ── Distributed initialisation ──────────────────────────────────────────────
#  theoetically used for multiple gpus i think
USE_DDP = "LOCAL_RANK" in os.environ  # set by torchrun

if USE_DDP:
    local_rank = int(os.environ["LOCAL_RANK"])
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)
    dist.init_process_group(backend="nccl", device_id=device)
    global_rank = dist.get_rank()
    world_size = dist.get_world_size()
    IS_MAIN = global_rank == 0  # only rank-0 prints / saves checkpoints
else:
    local_rank = 0
    global_rank = 0
    world_size = 1
    IS_MAIN = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if IS_MAIN:
    print(f"Mode      : {'DDP' if USE_DDP else 'Single-process'}")
    print(f"World size: {world_size}")
    print(f"Device    : {device}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# ── Paths ────────────────────────────────────────────────────────────────────
data_dir = Path("/scratch/cvcdt011/data")
crops_dir = data_dir / "crops"
trajectory_dir = data_dir / "rec1_trajectories"
CHECKPOINT_PATH = Path.cwd() / "best_convnext_tiny_bee_orientation.pth"

### 3. Index Samples & Split (shared loader)


In [ ]:
if IS_MAIN:
    print("Indexing crop samples from trajectory files...")

samples, _ = index_crops(crops_dir, trajectory_dir)
train_samples, val_samples, test_samples = split_by_trajectory(
    samples, val_fraction=0.15, test_fraction=0.15, seed=42
)

if IS_MAIN:
    print(
        f"Indexed {len(samples)} samples from {len({s.traj_id for s in samples})} trajectories."
    )
    for name, split in (
        ("train", train_samples),
        ("val", val_samples),
        ("test", test_samples),
    ):
        print(
            f"  {name}: {len(split)} samples, {len({s.traj_id for s in split})} trajectories"
        )

### 4. Datasets (shared `BeeOrientationDataset`)


In [ ]:
train_dataset = BeeOrientationDataset(train_samples, image_size=224)
val_dataset = BeeOrientationDataset(val_samples, image_size=224)
test_dataset = BeeOrientationDataset(test_samples, image_size=224)

if IS_MAIN:
    print(
        f"Datasets: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}"
    )

### 5. Hyperparameters and DataLoaders


In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
BATCH_SIZE_PER_GPU = 64  # per-GPU batch size; increase if VRAM allows
NUM_EPOCHS = 10
BASE_LR = 1e-4  # LR for a single GPU with batch_size=64
WEIGHT_DECAY = 1e-2
_cpus_total = 20
NUM_WORKERS = max(1, _cpus_total // world_size)
USE_AMP = torch.cuda.is_available()  # mixed precision
USE_COMPILE = torch.__version__ >= "2.0"  # torch.compile (PyTorch >= 2.0)

effective_batch_size = BATCH_SIZE_PER_GPU * world_size
scaled_lr = BASE_LR * (effective_batch_size / 64)

if IS_MAIN:
    print(f"Batch per GPU      : {BATCH_SIZE_PER_GPU}")
    print(f"Effective batch    : {effective_batch_size}")
    print(f"Scaled LR          : {scaled_lr:.2e}")
    print(f"AMP enabled        : {USE_AMP}")
    print(f"torch.compile      : {USE_COMPILE}")
    print(f"DataLoader workers : {NUM_WORKERS}")

# ── Samplers (DDP shards data across GPUs) ───────────────────────────────────
train_sampler = DistributedSampler(train_dataset, shuffle=True) if USE_DDP else None
val_sampler = DistributedSampler(val_dataset, shuffle=False) if USE_DDP else None
test_sampler = DistributedSampler(test_dataset, shuffle=False) if USE_DDP else No
_loader_kwargs = dict(
    batch_size=BATCH_SIZE_PER_GPU,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    multiprocessing_context="fork" if NUM_WORKERS > 0 else None,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)

train_loader = DataLoader(
    train_dataset,
    sampler=train_sampler,
    shuffle=(train_sampler is None),
    **_loader_kwargs,
)
val_loader = DataLoader(
    val_dataset, sampler=val_sampler, shuffle=False, **_loader_kwargs
)
test_loader = DataLoader(
    test_dataset, sampler=test_sampler, shuffle=False, **_loader_kwargs
)

if IS_MAIN:
    print(
        f"Train batches/GPU: {len(train_loader)} | Val batches/GPU: {len(val_loader)}"
    )

### 6. Model Initialization


In [ ]:
if IS_MAIN:
    print("Initializing ConvNeXt-Tiny model...")

model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

# Replace classifier head: 768 -> 2 (sin, cos)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, 2)

model = model.to(device)

if USE_COMPILE and torch.cuda.is_available():
    model = torch.compile(model, mode="reduce-overhead")
    if IS_MAIN:
        print("torch.compile(mode='reduce-overhead') enabled")

# ── Wrap for multi-GPU ────────────────────────────────────────────────────────
if USE_DDP:
    model = DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank,
        gradient_as_bucket_view=True,
    )
    if IS_MAIN:
        print(f"DDP: wrapped model across {world_size} GPUs")
elif torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    if IS_MAIN:
        print(f"DataParallel: using {torch.cuda.device_count()} GPUs")
else:
    if IS_MAIN:
        print(f"Single device: {device}")


### 7. Loss, Optimizer, Scaler, and Scheduler


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=scaled_lr, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# GradScaler for automatic mixed precision -- no-op when USE_AMP=False
scaler = torch.amp.GradScaler(enabled=USE_AMP)

### 8. Custom Angular Metric


In [ ]:
def compute_angular_error_deg(pred, target):
    """Computes Mean Absolute Error (MAE) in degrees using circular wrapping."""
    pred_rad = torch.atan2(pred[:, 0], pred[:, 1])
    target_rad = torch.atan2(target[:, 0], target[:, 1])

    diff_rad = pred_rad - target_rad
    diff_rad = torch.atan2(torch.sin(diff_rad), torch.cos(diff_rad))

    diff_deg = torch.abs(torch.rad2deg(diff_rad))
    return diff_deg.mean().item()

### 9. Training and Validation Step Helpers


In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    model.train()

    # DDP: update sampler seed so each GPU sees a different shuffle each epoch
    if USE_DDP and hasattr(loader.sampler, "set_epoch"):
        loader.sampler.set_epoch(epoch)

    # Accumulate as CUDA tensors so we can all_reduce across ranks cleanly
    running_loss = torch.tensor(0.0, device=device)
    running_mae = torch.tensor(0.0, device=device)
    n_samples = torch.tensor(0, device=device, dtype=torch.long)

    pbar = tqdm(loader, desc=f"Train E{epoch + 1}", leave=False, disable=not IS_MAIN)
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        bs = images.size(0)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda" if torch.cuda.is_available() else "cpu", enabled=USE_AMP
        ):
            outputs = model(images)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.detach() * bs
        running_mae += (
            compute_angular_error_deg(outputs.detach().float(), targets.float()) * bs
        )
        n_samples += bs

    # Sum metrics across all DDP ranks so every rank gets the global average
    if USE_DDP:
        dist.all_reduce(running_loss, op=dist.ReduceOp.SUM)
        dist.all_reduce(running_mae, op=dist.ReduceOp.SUM)
        dist.all_reduce(n_samples, op=dist.ReduceOp.SUM)

    epoch_loss = (running_loss / n_samples).item()
    epoch_mae = (running_mae / n_samples).item()
    return epoch_loss, epoch_mae


def validate(model, loader, criterion, device):
    model.eval()

    running_loss = torch.tensor(0.0, device=device)
    running_mae = torch.tensor(0.0, device=device)
    n_samples = torch.tensor(0, device=device, dtype=torch.long)

    with torch.no_grad():
        pbar = tqdm(loader, desc="Validating", leave=False, disable=not IS_MAIN)
        for images, targets in pbar:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            bs = images.size(0)

            with torch.amp.autocast(
                device_type="cuda" if torch.cuda.is_available() else "cpu",
                enabled=USE_AMP,
            ):
                outputs = model(images)
                loss = criterion(outputs, targets)

            running_loss += loss.detach() * bs
            running_mae += (
                compute_angular_error_deg(outputs.float(), targets.float()) * bs
            )
            n_samples += bs

    if USE_DDP:
        dist.all_reduce(running_loss, op=dist.ReduceOp.SUM)
        dist.all_reduce(running_mae, op=dist.ReduceOp.SUM)
        dist.all_reduce(n_samples, op=dist.ReduceOp.SUM)

    val_loss = (running_loss / n_samples).item()
    val_mae = (running_mae / n_samples).item()
    return val_loss, val_mae


### 10. Main Training Loop


In [ ]:
best_val_loss = float("inf")
history = {"train_loss": [], "train_mae": [], "val_loss": [], "val_mae": []}

if IS_MAIN:
    print("Starting training...")
    print(f"  GPUs        : {world_size}")
    print(f"  Epochs      : {NUM_EPOCHS}")
    print(f"  LR (scaled) : {scaled_lr:.2e}")
    print(f"  AMP         : {USE_AMP}")


def unwrap_model(m):
    """Strip DDP / DataParallel / torch.compile wrappers to get the raw module."""
    if isinstance(m, (DDP, nn.DataParallel)):
        m = m.module
    # torch.compile may nest _orig_mod more than once -- loop to be safe
    while hasattr(m, "_orig_mod"):
        m = m._orig_mod
    return m


for epoch in range(NUM_EPOCHS):
    train_loss, train_mae = train_epoch(
        model, train_loader, criterion, optimizer, scaler, device, epoch
    )
    val_loss, val_mae = validate(model, val_loader, criterion, device)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_mae"].append(train_mae)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)

    if IS_MAIN:
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}:")
        print(f"  Train Loss (MSE): {train_loss:.4f} | Train MAE: {train_mae:.2f}°")
        print(f"  Val   Loss (MSE): {val_loss:.4f}   | Val   MAE: {val_mae:.2f}°")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            raw_model = unwrap_model(model)
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": raw_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": best_val_loss,
                },
                CHECKPOINT_PATH,
            )
            print("  --> Saved best model checkpoint!")

    # Synchronise processes at end of each epoch
    if USE_DDP:
        dist.barrier()

if USE_DDP:
    dist.destroy_process_group()


### 11. Plot Training History


In [ ]:
if IS_MAIN:
    epochs_range = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history["train_loss"], label="Train Loss")
    plt.plot(epochs_range, history["val_loss"], label="Val Loss")
    plt.title("MSE Loss vs Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history["train_mae"], label="Train MAE (\u00b0)")
    plt.plot(epochs_range, history["val_mae"], label="Val MAE (\u00b0)")
    plt.title("Mean Absolute Angular Error vs Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("MAE (Degrees)")
    plt.legend()

    plt.tight_layout()
    plt.savefig(Path.cwd() / "summary.png", dpi=150)
    plt.show()

### 12. Evaluate on Test Split


In [ ]:
if IS_MAIN:
    # Reload a clean model for evaluation
    eval_model = convnext_tiny()
    in_features = eval_model.classifier[2].in_features
    eval_model.classifier[2] = nn.Linear(in_features, 2)

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    state_dict = checkpoint["model_state_dict"]
    _PREFIXES = ("module.", "_orig_mod.")
    changed = True
    while changed:
        changed = False
        for prefix in _PREFIXES:
            if any(k.startswith(prefix) for k in state_dict):
                state_dict = {k.removeprefix(prefix): v for k, v in state_dict.items()}
                changed = True

    eval_model.load_state_dict(state_dict)
    eval_model = eval_model.to(device)

    test_loss, test_mae = validate(eval_model, test_loader, criterion, device)
    print(f"\nFinal Test Results (checkpoint from epoch {checkpoint['epoch']}):")
    print(f"  Test Loss (MSE): {test_loss:.4f}")
    print(f"  Test MAE       : {test_mae:.2f}°")
